# PATSTAT inventors — `inventor_list`, `n_inventors`, first and last inventor

One row per application in the universe with at least one inventor, keyed on `appln_id` like every other
file in `PATSTAT/output/`. The twin of `PatentView/notebook/patent_inventor.ipynb` (and so of the paper
side's `paper_author.parquet`); `patstat_inventor_country.parquet` is its geography companion and shares
`n_inventors`.

## Raw data
```
raw/tls207  pers_appln   person_id, appln_id, invt_seq_nr (> 0: an inventor, in the order printed), applt_seq_nr
raw/tls206  person       person_name (empty = an unnamed placeholder record)
PATSTAT/output/patstat_metadata.parquet   the universe (ps.UNIVERSE_WHERE); inventor_list, the cross-check
```

## What an inventor id is here — and is not
`person_id` is PATSTAT's key for one person record (a name + address as the office recorded it). It is
**not** a disambiguated inventor: the same person on two applications usually has two `person_id`s, so
unlike PatentView's `inventor_id` it cannot follow an inventor across applications. Within one
application it is exact, which is what `inventor_list`, `n_inventors` and the first / last inventor need.

## One slot per person — and the unnamed placeholder
tls207 is **not** unique on `(person_id, appln_id)`: the same person can be listed at several sequence
numbers of one application. Two different things cause it, and they are treated differently:
- a **named** person listed twice (e.g. `KORITZ,DIETER,DD` at inventor positions 1 and 7) is one inventor —
  kept once, at the lowest sequence (PatentView's rule);
- an **unnamed** person record (`tls206.person_name` empty: person_id 263 `-NOT AVAILABLE-` and 83 others) is a
  placeholder that stands for several unrelated people — on 14,030 applications it fills more than one
  inventor slot — so each of its slots counts as a separate inventor and its id can repeat in the list.

`patstat_metadata.inventor_list`, this file and `patstat_inventor_country.n_inventors` all apply this rule.

Order within the list: `invt_seq_nr`, then `person_id`.

## Output — `PATSTAT/output/patstat_inventor.parquet`
| column | |
|---|---|
| `appln_id` | application in the universe with ≥ 1 inventor row |
| `inventor_list` | inventor `person_id`s in sequence order, `;`-joined (ints, so a plain split is safe; an unnamed placeholder id can repeat) |
| `n_inventors` | distinct inventors — `paper_author.team_size` under its patent name |
| `first_inventor` / `last_inventor` | `person_id` at the lowest / highest sequence |

> **Clock switch.** This text describes the default filing clock (`output/`). With `NB_PS_CLOCK=grant` the same notebook runs on the grant clock, PatentView's: years and ages are grant years (first publication with `publn_first_grant = 'Y'`), only applications with a grant year are in the universe (both ends of every edge), year / age columns are named `grant_year` / `yrs_since_grant`, and the outputs go to `PATSTAT/output_grant/` (executed copy: `notebook/grant/`).

> **Family unit.** With `NB_PS_UNIT=family` the same notebook runs with the DOCDB family as the unit (`docdb_family_id`), dated by the family's earliest priority year (`priority_year`, `yrs_since_priority`): citations are distinct family-to-family pairs with within-family citations dropped, classification comes from the earliest member that has one, inventors and their countries from one representative member (the one with the most located inventors), and the outputs go to `PATSTAT/output_family/` (executed copy: `notebook/family/`).


In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP     = ps.out('patstat_inventor.parquet')
META_FP    = ps.out('patstat_metadata.parquet')
COUNTRY_FP = ps.out('patstat_inventor_country.parquet')    # cross-check if it already exists
ps.preflight('patstat_inventor')
con = ps.connect()

dump   : /project/jevans/PATSTAT/unzipped_data   (65 zip parts, 11 tables registered)
raw    : /project/jevans/Dawoon/Science of Science/PATSTAT/raw
output : /project/jevans/Dawoon/Science of Science/PATSTAT/output

  patstat_inventor              OK


## 1. Build

In [2]:
%%time
t0 = time.time()
# (appln_id, doc): the applications whose inventors are read and the unit they count for -- each application
# itself, or (NB_PS_UNIT=family) the representative member inv_appln_id chosen in patstat_metadata
DOCS = (f"SELECT inv_appln_id AS appln_id, docdb_family_id AS doc FROM read_parquet('{META_FP}') WHERE inv_appln_id IS NOT NULL"
        if ps.UNIT == 'family' else f"SELECT appln_id, appln_id AS doc FROM read_parquet('{META_FP}')")
PH = [r[0] for r in con.execute(f"SELECT person_id FROM {ps.raw('tls206')} WHERE trim(person_name) = '' ORDER BY 1").fetchall()]
print(f'{len(PH)} unnamed person records (placeholders), e.g. {PH[:5]}')
con.execute(f"""CREATE OR REPLACE TABLE inv AS
  WITH r AS (SELECT dm.doc, pa.person_id, pa.invt_seq_nr, list_contains({PH}, pa.person_id) AS unnamed
             FROM {ps.raw('tls207')} pa JOIN ({DOCS}) dm USING (appln_id)
             WHERE pa.invt_seq_nr > 0),
       d AS (SELECT doc, person_id, min(invt_seq_nr) AS seq, bool_or(unnamed) AS unnamed FROM r
             GROUP BY doc, person_id, CASE WHEN unnamed THEN invt_seq_nr END)
  SELECT *, row_number() OVER (PARTITION BY doc ORDER BY seq, person_id) AS pos FROM d""")
n_rows = con.execute('SELECT count(*), count(DISTINCT doc), count(*) FILTER (WHERE unnamed) FROM inv').fetchone()
print(f'{n_rows[0]:,} inventor slots on {n_rows[1]:,} {ps.UNITS} ({n_rows[2]:,} unnamed)   [{time.time()-t0:.0f}s]')
con.execute(f"""COPY (
  SELECT doc AS {ps.KEY},
         string_agg(person_id::VARCHAR, ';' ORDER BY pos) AS inventor_list,
         count(*)::INTEGER                                AS n_inventors,
         arg_min(person_id, pos)                          AS first_inventor,
         arg_max(person_id, pos)                          AS last_inventor
  FROM inv GROUP BY doc ORDER BY doc
) TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
n = pq.ParquetFile(OUT_FP).metadata.num_rows
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e6:.1f} MB, {n:,} {ps.UNITS}) in {time.time()-t0:.0f}s')

84 unnamed person records (placeholders), e.g. [263, 5312968, 5594388, 6614142, 7048592]
195,446,903 inventor slots on 75,617,406 applications (23,668 unnamed)   [14s]
WROTE /project/jevans/Dawoon/Science of Science/PATSTAT/output/patstat_inventor.parquet  (1385.2 MB, 75,617,406 applications) in 66s


## 2. Verification

1. `appln_id` unique; no nulls; `n_inventors == len(inventor_list.split(';'))`; no duplicate **named** id in a list;
   `first_inventor` / `last_inventor` are its two ends (taken by `arg_min` / `arg_max`, a different
   aggregate from the `string_agg`, so the check is not a tautology).
2. Against `patstat_metadata.inventor_list`: identical strings on every shared application, and the same
   set of applications (same table, same order rule).
3. Against `patstat_inventor_country.n_inventors`, if that file exists already.
4. Team-size distribution, by filing decade, and the largest teams.

In [3]:
%%time
chk = con.execute(f"""
WITH y AS (SELECT *, str_split(inventor_list, ';') AS parts FROM read_parquet('{OUT_FP}')),
     x AS (SELECT *, list_filter(parts, z -> NOT list_contains({[str(p) for p in PH]}, z)) AS named FROM y)
SELECT count(*)                                                        AS applications,
       count(*) - count(DISTINCT {ps.KEY})                             AS dup_appln_id,
       count(*) FILTER (WHERE inventor_list IS NULL OR first_inventor IS NULL OR last_inventor IS NULL) AS nulls,
       sum(n_inventors)                                                AS inventor_slots,
       count(*) FILTER (WHERE n_inventors <> len(parts))               AS size_mismatch,
       count(*) FILTER (WHERE len(named) <> len(list_distinct(named)))  AS dup_named_in_list,
       count(*) FILTER (WHERE len(named) < len(parts))                 AS with_unnamed,
       count(*) FILTER (WHERE parts[1] <> first_inventor::VARCHAR)     AS first_mismatch,
       count(*) FILTER (WHERE parts[len(parts)] <> last_inventor::VARCHAR) AS last_mismatch,
       count(*) FILTER (WHERE n_inventors = 1)                         AS solo,
       max(n_inventors)                                                AS max_inventors
FROM x""").fetchdf().iloc[0]
for k, v in chk.items():
    print(f'  {k:<18} {int(v):>14,}')
ok = all(chk[k] == 0 for k in ['dup_appln_id', 'nulls', 'size_mismatch', 'dup_named_in_list', 'first_mismatch', 'last_mismatch'])
print(f"\n1.   {'ALL PASS' if ok else 'FAILED'}   (mean team size {chk.inventor_slots/chk.applications:.3f}, solo {chk.solo/chk.applications*100:.1f}%)")

m = con.execute(f"""
WITH n AS (SELECT {ps.KEY}, inventor_list FROM read_parquet('{OUT_FP}')),
     m AS (SELECT {ps.KEY}, inventor_list FROM read_parquet('{META_FP}') WHERE inventor_list IS NOT NULL)
SELECT (SELECT count(*) FROM n JOIN m USING ({ps.KEY}))                                          AS shared,
       (SELECT count(*) FROM n JOIN m USING ({ps.KEY}) WHERE n.inventor_list <> m.inventor_list) AS list_mismatch,
       (SELECT count(*) FROM n ANTI JOIN m USING ({ps.KEY}))                                     AS only_here,
       (SELECT count(*) FROM m ANTI JOIN n USING ({ps.KEY}))                                     AS only_metadata""").fetchdf().iloc[0]
print(f'\n2.   vs patstat_metadata.inventor_list: ' + '  '.join(f'{k} {int(v):,}' for k, v in m.items()))
ok2 = m.list_mismatch == 0 and m.only_metadata == 0
print(f"     {'OK' if ok2 else 'FAILED'}" + ('' if m.only_here == 0 else
      f'   ({int(m.only_here):,} applications have inventors here but none in metadata: an inventor person_id missing from tls206)'))
if os.path.exists(COUNTRY_FP):
    c = con.execute(f"""SELECT count(*) AS shared, count(*) FILTER (WHERE n.n_inventors <> c.n_inventors) AS n_mismatch
                        FROM read_parquet('{OUT_FP}') n JOIN read_parquet('{COUNTRY_FP}') c USING ({ps.KEY})""").fetchdf().iloc[0]
    print(f'3.   vs patstat_inventor_country.n_inventors: {int(c.shared):,} shared, {int(c.n_mismatch):,} differ')
else:
    print('3.   patstat_inventor_country.parquet not present yet -- it checks this file instead')

print('\n4.   team-size distribution')
display(con.execute(f"""SELECT CASE WHEN n_inventors >= 10 THEN '10+' ELSE lpad(n_inventors::VARCHAR, 2, ' ') END AS n_inventors,
                        count(*) AS applications, round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
                        FROM read_parquet('{OUT_FP}') GROUP BY 1 ORDER BY 1""").fetchdf())
print('     by filing decade')
display(con.execute(f"""SELECT (m.filing_year // 10) * 10 AS decade, count(*) AS applications,
       round(avg(s.n_inventors), 3) AS mean_inventors,
       round(100.0 * count(*) FILTER (WHERE s.n_inventors = 1) / count(*), 1) AS pct_solo,
       round(100.0 * count(*) FILTER (WHERE s.n_inventors >= 5) / count(*), 1) AS pct_5plus
FROM read_parquet('{OUT_FP}') s JOIN read_parquet('{META_FP}') m USING ({ps.KEY}) GROUP BY 1 ORDER BY 1""").fetchdf())
display(con.execute(f"SELECT {ps.KEY}, n_inventors, first_inventor, last_inventor FROM read_parquet('{OUT_FP}') ORDER BY n_inventors DESC, {ps.KEY} LIMIT 5").fetchdf())
con.close()
assert ok and ok2

  applications           75,617,406
  dup_appln_id                    0
  nulls                           0
  inventor_slots        195,446,903
  size_mismatch                   0
  dup_named_in_list               0
  with_unnamed               14,032
  first_mismatch                  0
  last_mismatch                   0
  solo                   30,647,745
  max_inventors                 133

1.   ALL PASS   (mean team size 2.585, solo 40.5%)

2.   vs patstat_metadata.inventor_list: shared 75,617,406  list_mismatch 0  only_here 0  only_metadata 0
     OK
3.   patstat_inventor_country.parquet not present yet -- it checks this file instead

4.   team-size distribution


,n_inventors,applications,pct
0,1,30647745,40.53
1,2,16017927,21.18
2,3,11271988,14.91
3,4,7068622,9.35
4,5,4272424,5.65
5,6,2499499,3.31
6,7,1408839,1.86
7,8,876761,1.16
8,9,513937,0.68
9,10+,1039664,1.37


     by filing decade


,decade,applications,mean_inventors,pct_solo,pct_5plus
0,1900,674364,1.142,87.5,0.0
1,1910,691920,1.125,88.8,0.0
2,1920,698080,1.121,89.3,0.0
3,1930,697353,1.162,86.1,0.1
4,1940,627594,1.194,83.7,0.1
5,1950,1018012,1.343,74.5,0.5
6,1960,1551627,1.541,64.6,1.4
7,1970,3377906,1.996,50.1,5.8
8,1980,7057108,2.214,44.2,8.2
9,1990,9918257,2.289,42.5,9.1


,appln_id,n_inventors,first_inventor,last_inventor
0,478696789,133,12495599,54975134
1,479859931,133,47025612,47093343
2,486242500,133,47225249,72277774
3,503087612,133,57549022,63977097
4,503350846,133,12495599,54975134
